# Lab 07 — 07 Freshness & Volume



In [ ]:
from pathlib import Path
import sys
cwd=Path.cwd().resolve()
project_root=next((p for p in [cwd,*cwd.parents] if (p/'src'/'lab07').exists()),None)
if project_root and str(project_root/'src') not in sys.path: sys.path.insert(0,str(project_root/'src'))
if project_root and str(project_root/'tools') not in sys.path: sys.path.insert(0,str(project_root/'tools'))
dbutils.widgets.text('catalog','dbr_dev','01 Catalog'); dbutils.widgets.text('schema','parvinbadalov','02 Schema'); dbutils.widgets.text('volume_name','lab07_data_quality','03 Volume'); dbutils.widgets.text('run_id','manual','04 Run ID')
catalog=dbutils.widgets.get('catalog'); schema=dbutils.widgets.get('schema'); volume_name=dbutils.widgets.get('volume_name'); run_id=dbutils.widgets.get('run_id')
assert catalog=='dbr_dev' and schema=='parvinbadalov', f'Lab 07 requires dbr_dev.parvinbadalov, got {catalog}.{schema}'
volume_root=f'/Volumes/{catalog}/{schema}/{volume_name}'


In [ ]:
from pyspark.sql import functions as F
from lab07.reconciliation import percent_change,anomaly_status
x=spark.table(f'{catalog}.{schema}.business_license_landing'); rows=x.count(); updated=x.agg(F.max(F.to_timestamp('_source_dataset_updated_at'))).first()[0]; assert updated is not None
hours=spark.sql(f"SELECT (unix_timestamp(current_timestamp())-unix_timestamp(timestamp'{updated}'))/3600.0 h").first()['h']; print('freshness hours',hours); assert hours<=72
hist=f'{catalog}.{schema}.lab07_volume_history'; baseline=rows
if spark.catalog.tableExists(hist):
    prev=spark.table(hist).orderBy(F.desc('recorded_at')).limit(1).collect(); baseline=int(prev[0]['row_count']) if prev else rows
change=percent_change(rows,baseline); status,severity=anomaly_status(change,25,50); print('volume',change,status); assert status!='FAIL'
spark.createDataFrame([(run_id,rows,baseline,float(change))],'run_id string,row_count long,baseline_row_count long,change_pct double').withColumn('recorded_at',F.current_timestamp()).write.mode('append').saveAsTable(hist)
